# L29 Lab: Range-Doppler and MTI, notebook

This notebook runs the course's radar chain, `radar_lab.py`, on the bursts you saved in the Phaser app. You do not write the chain; you run it, change what each task names, and record what you see in Table F of the L29 Lab packet. There are three tasks, (a) to (c), about 30 to 45 minutes in all, and about five changed lines.

**To run it:** put this notebook in the folder you made for the L28 Lab, beside `radar_lab.py` and your captures, or keep the course's layout (`notebooks/` beside `radar-data/`). Run the cells in order from the top.

If your own bursts failed, the course's recorded data (`L29_*.npz`) works in every cell.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import radar_lab as rl

# Where the captures are: the course's recorded data if this folder sits beside
# radar-data/, otherwise this folder.
DATA = Path("../radar-data") if Path("../radar-data").exists() else Path(".")

# Your hand burst with the wall in view (Procedures B and E). Change the name to your own file.
CAPTURE = DATA / "L29_hand_closing.npz"
cap = rl.load_capture(CAPTURE)
lam = rl.C / cap["fc"]
n_chirps, n_samp = cap["iq"].shape
print(f"{CAPTURE.name}: {n_chirps} chirps of {n_samp} samples, B = {cap['B']/1e6:.0f} MHz, "
      f"T_ramp = {cap['T_ramp']*1e6:.0f} us, PRI = {cap['PRI']*1e3:.2f} ms, f0 = {cap['f0']/1e3:.1f} kHz")
print(f"cells: {rl.C/(2*cap['B']):.2f} m in range, {lam/(2*n_chirps*cap['PRI']):.3f} m/s in velocity; "
      f"velocity limit +/-{lam/(4*cap['PRI']):.1f} m/s")

In [ ]:
# Readouts used by the tasks. Run this cell; you do not need to change it.
def strongest(rd, r, v, r_lo, r_hi, v_lo, v_hi):
    """Range, velocity, and level of the strongest cell in a box of the map."""
    m = ((r >= r_lo) & (r <= r_hi))[:, None] & ((v >= v_lo) & (v <= v_hi))[None, :]
    i, j = np.unravel_index(np.argmax(np.where(m, rd, -400)), rd.shape)
    return i, j, rd[i, j]

def mover(rd, r, v, r_lo=0.3, r_hi=6.0):
    """The strongest cell moving faster than 0.3 m/s either way, between r_lo and r_hi m."""
    return max(strongest(rd, r, v, r_lo, r_hi, 0.3, 15), strongest(rd, r, v, r_lo, r_hi, -15, -0.3),
               key=lambda c: c[2])

def peak_velocity(row, j, v):
    """Velocity of the peak at index j of row, between cells: a parabola through it and its neighbors in dB."""
    a, b, c = row[j - 1], row[j], row[(j + 1) % len(row)]
    return v[j] + 0.5 * (a - c) / (a - 2 * b + c) * (v[1] - v[0])

def floor(rd, r, v, v_lo=-15, v_hi=15):
    """Median level of the map from 6 to 20 m, between v_lo and v_hi m/s: the noise floor there."""
    return np.median(rd[(r > 6) & (r < 20)][:, (v >= v_lo) & (v <= v_hi)])

def width_3db(row, j, step):
    """Width 3 dB down of the peak at index j of row, interpolated between cells."""
    top = row[j] - 3
    lo = hi = j
    while lo > 0 and row[lo - 1] > top: lo -= 1
    while hi < len(row) - 1 and row[hi + 1] > top: hi += 1
    left = lo - (row[lo] - top) / (row[lo] - row[lo - 1])
    right = hi + (row[hi] - top) / (row[hi] - row[hi + 1])
    return (right - left) * step

def show(rd, r, v, title, r_max=4.0, v_max=5.0, vmin=-120, vmax=-20, ax=None):
    """Draw a range-Doppler map, range up and velocity across, positive closing to the right."""
    ax = ax or plt.figure(figsize=(7, 3.5)).gca()
    keep = r <= r_max
    im = ax.pcolormesh(v, r[keep], rd[keep], vmin=vmin, vmax=vmax, cmap="Blues", shading="auto")
    ax.set_xlim(-v_max, v_max)
    ax.set_xlabel("velocity (m/s), + closing"); ax.set_ylabel("range (m)"); ax.set_title(title)
    plt.colorbar(im, ax=ax, label="dBFS")
    return ax

## Your range-Doppler map

No task here. The cell makes the map of your burst with `rl.range_doppler` and reads two cells from it: the strongest thing that moves (faster than 0.3 m/s either way) and the strongest thing that does not (0 m/s, beyond 1.5 m), which is the wall in the recorded data.

**Check:** the recorded hand reads about +1.0 m/s at about 1.0 m, positive because it closes; the wall reads 0 m/s at 2.5 m. Your own hand and wall should land where you put them, to within a cell. If you saved a 200 MHz burst in Procedure D, set `CAPTURE` to it and run this cell again: the hand's spot is 2.5 times taller in range.

In [ ]:
rd, r, v = rl.range_doppler(cap)
i_hand, j, lvl = mover(rd, r, v)
iw, jw, lw = strongest(rd, r, v, 1.5, 6.0, -0.01, 0.01)
print(f"strongest mover: {r[i_hand]:.2f} m, {peak_velocity(rd[i_hand], j, v):+.2f} m/s, {lvl:.1f} dBFS")
print(f"strongest static return beyond 1.5 m: {r[iw]:.2f} m, {lw:.1f} dBFS")
print(f"noise floor: {floor(rd, r, v):.1f} dBFS")
show(rd, r, v, CAPTURE.name); plt.show()

## Task (a): the chirp count sets the velocity cell

The cell keeps only the first `M` chirps of the burst, as the Phaser app does when you lower its chirp count, makes the map, and reads, in the hand's range cell from the map above, the hand's velocity, its level, the width of its peak 3 dB down in velocity, and the noise floor.

1. Run it as it stands (`M = 256`) and record the row in Table F.
2. **Change one line:** set `M = 128`, run it, and record the row. Then `M = 64`.

**Check:** the velocity cell, printed as `dv`, is $\lambda/(2M\,T_\text{PRI})$: 0.117, 0.234, and 0.469 m/s. The hand's peak is about one to one and a half cells wide at every $M$, so its width in m/s doubles each time you halve $M$. Its level stays within about half a decibel, because `radar_lab` divides each FFT by the sum of its window, while the noise floor rises 3 dB each time: half the chirps, half the integration gain of Lesson 27.

In [ ]:
M = 256                  # TASK (a): change to 128, then 64

short = {**cap, "iq": cap["iq"][:M]}
rd_m, r_m, v_m = rl.range_doppler(short)
dv = v_m[1] - v_m[0]
i = i_hand                                    # the hand's range cell, from the full burst's map
j = int(np.argmax(np.where(np.abs(v_m) > 0.3, rd_m[i], -400)))
lvl = rd_m[i, j]
print(f"M = {M}: dv = {dv:.3f} m/s (lambda/(2 M PRI) = {lam/(2*M*cap['PRI']):.3f}); "
      f"hand {peak_velocity(rd_m[i], j, v_m):+.2f} m/s at {lvl:.1f} dBFS, {width_3db(rd_m[i], j, dv):.2f} m/s wide 3 dB down; "
      f"floor {floor(rd_m, r_m, v_m):.1f} dBFS")

plt.figure(figsize=(7, 3))
plt.plot(v_m, rd_m[i], ".-")
plt.xlim(-5, 5); plt.ylim(-125, -20)
plt.xlabel("velocity (m/s), + closing"); plt.ylabel("power (dBFS)")
plt.title(f"the hand's range cell, {r_m[i]:.2f} m, M = {M}"); plt.grid(alpha=0.3); plt.show()

## Task (b): 2-pulse against 3-pulse

The cell runs the burst through `rl.mti` and compares the map with the one that has no canceller. It reads the wall's cell, the hand, and the noise floor in two places: in the hand's own velocity column, and near 14 m/s, the edge of the velocity window.

1. **Change one line, step 1:** replace `None` in `H_DB = None` with the canceller's predicted power gain at the hand's velocity, in dB, from `f_D`, `cap["PRI"]`, and `STAGES`, the number of 2-pulse stages in `KIND` (the gain of the lesson's canceller-response frame). Run the cell with `KIND = "2pulse"` and record the row.
2. **Change one line, step 2:** set `KIND = "3pulse"` and run it again.

**Check:** the hand drops by your `H_DB`, to within about half a decibel: about 13.6 dB for the 2-pulse canceller and 27.2 dB for the 3-pulse at 1.0 m/s. The wall falls into the floor; in the recorded data that is more than 100 dB, because the simulated chirps are timed perfectly, and on your kit it is however far chirp sync and the room allow. The floor does not rise by the same amount everywhere: the noise passes through the same filter as the targets, so near 14 m/s it rises about 6 and 12 dB, and in the hand's own column it falls with the hand. Part 3 asks what that does to the hand's SNR.

In [ ]:
KIND = "2pulse"          # TASK (b), step 2: change to "3pulse"
STAGES = {"none": 0, "2pulse": 1, "3pulse": 2}[KIND]

rd0, r0, v0 = rl.range_doppler(cap)
rd1, r1, v1 = rl.range_doppler(rl.mti(cap, KIND))
i, j, h0 = mover(rd0, r0, v0)
v_hand = peak_velocity(rd0[i], j, v0)
h1 = rd1[i, np.argmin(np.abs(v1 - v0[j]))]
f_D = 2 * v_hand / lam                       # the hand's Doppler frequency, Hz

H_DB = None              # TASK (b), step 1: predicted power gain at f_D, in dB, for STAGES stages

iw, jw, w0 = strongest(rd0, r0, v0, 1.5, 6.0, -0.01, 0.01)
w1 = rd1[iw, np.argmin(np.abs(v1))]
col = lambda rd, v, vv: floor(rd, r0, v, vv - 0.15, vv + 0.15)
print(f"{KIND}: hand at {v_hand:+.2f} m/s, f_D = {f_D:.1f} Hz: {h0:.1f} -> {h1:.1f} dBFS, "
      f"change {h1 - h0:+.1f} dB; predicted {H_DB if H_DB is None else round(H_DB, 1)}")
print(f"wall at {r0[iw]:.2f} m: {w0:.1f} -> {w1:.1f} dBFS, down {w0 - w1:.0f} dB")
print(f"floor in the hand's column: {col(rd0, v0, v_hand):.1f} -> {col(rd1, v1, v_hand):.1f} dBFS; "
      f"near 14 m/s: {floor(rd0, r0, v0, 13.5, 14.5):.1f} -> {floor(rd1, r0, v1, 13.5, 14.5):.1f} dBFS")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
show(rd0, r0, v0, "no MTI", ax=ax[0]); show(rd1, r0, v1, KIND, ax=ax[1])
plt.tight_layout(); plt.show()

## Task (c): the spinner's spectrogram, kept for L31

The map shows the fidget spinner as a band of velocities at one range. A spectrogram cuts one range cell's slow-time signal into short pieces and takes the FFT of each, so it shows how that band changes during the burst. `rl.spectrogram` subtracts the cell's mean first, which removes the hub and the hand that holds it.

1. Run the first cell below to see the spinner's map. **Change one line:** set `R_SPIN` in the second cell to the range of the band, read from the map, and run it with `N_WIN = 64`. Record the row in Table F.
2. **Change one line:** set `N_WIN = 32` and run it again. Record the row.

**Check:** the recorded spinner sits at 0.47 m and its band reaches about ±3 m/s at both window lengths; the tip speed is $2\pi(0.03\ \text{m})(15\ \text{rev/s}) = 2.83$ m/s. Halving `N_WIN` doubles the Doppler cell, from 0.47 to 0.94 m/s, and halves the time between columns, from 8 to 4 ms: the trade-off Lesson 31 starts from. The last cell saves the spectrogram for the L31 Lab.

In [ ]:
SPIN = DATA / "L29_spinner.npz"   # change to your own spinner burst (Procedure C)
spin = rl.load_capture(SPIN)
rd_s, r_s, v_s = rl.range_doppler(spin)
show(rd_s, r_s, v_s, SPIN.name, r_max=2.0); plt.show()

In [ ]:
R_SPIN = None            # TASK (c), step 1: the spinner's range from the map, in m
N_WIN = 64               # TASK (c), step 2: change to 32
HOP = N_WIN // 4

if R_SPIN is None:
    print("Set R_SPIN to the range of the spinner's band on the map above, then run this cell again.")
else:
    k = int(np.argmin(np.abs(rl.range_axis(spin) - R_SPIN)))     # the range FFT bin, as spectrogram wants it
    t, f, S = rl.spectrogram(spin, k, n_win=N_WIN, hop=HOP)
    vel = f * lam / 2
    band = vel[10 * np.log10(np.mean(10 ** (S / 10), axis=1)) - 10 * np.log10(np.mean(10 ** (S / 10), axis=1)).max() > -10]
    print(f"bin {k}, {rl.range_axis(spin)[k]:.2f} m: Doppler cell {vel[1] - vel[0]:.2f} m/s, "
          f"columns {(t[1] - t[0]) * 1e3:.0f} ms apart; band within 10 dB of its peak: "
          f"{band.min():+.2f} to {band.max():+.2f} m/s")
    plt.figure(figsize=(7, 3.5))
    plt.pcolormesh(t * 1e3, vel, S, vmin=-40, vmax=0, cmap="Blues", shading="auto")
    plt.xlabel("time (ms)"); plt.ylabel("velocity (m/s), + closing")
    plt.title(f"{SPIN.name}, {rl.range_axis(spin)[k]:.2f} m, N_WIN = {N_WIN}")
    plt.colorbar(label="dB"); plt.show()

In [ ]:
# Keep for the L31 Lab: the spectrogram you just made, with what made it.
if R_SPIN is not None:
    np.savez("L29_spinner_spectrogram.npz", t=t, f=f, S_db=S, range_m=rl.range_axis(spin)[k],
             n_win=N_WIN, hop=HOP, capture=SPIN.name)
    print("saved L29_spinner_spectrogram.npz")

## Before you close

Save the notebook with your changes and your plots, and turn it in with the packet. Keep it, `radar_lab.py`, your bursts, and `L29_spinner_spectrogram.npz` in the Module 4 folder: the L30 Lab runs CFAR on the maps of these bursts, with and without MTI, and the L31 Lab starts from the spectrogram. Anything you do beyond these three tasks, such as another capture, another range cell, or the 3-pulse canceller on the spinner, belongs in your report's Extra Effort.